# 03 - Análisis de nulos y mecanismos de ausencia

**Objetivo.** Analizar los faltantes luego de la normalización inicial, distinguiendo faltantes estructurales por fuente de patrones potencialmente informativos dentro de cada dataset.

Este notebook agrega flags `*_is_null`, genera matrices faltante-vs-faltante, matrices faltante-vs-cuantitativas y gráficos de porcentaje de faltantes por label categórico.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de datasets preprocesados

In [ ]:
PROCESSED_SOURCES = {
    'df_preprocesado_meli_alq': PROCESSED_DIR / 'df_preprocesado_meli_alq_v1.csv',
    'df_preprocesado_meli_alq_temp': PROCESSED_DIR / 'df_preprocesado_meli_alq_temp_v1.csv',
    'df_preprocesado_meli_vtas': PROCESSED_DIR / 'df_preprocesado_meli_vtas_v1.csv',
    'df_preprocesado_argenprop': PROCESSED_DIR / 'df_preprocesado_argenprop_v1.csv',
    'df_preprocesado_zonaprop': PROCESSED_DIR / 'df_preprocesado_zonaprop_v1.csv',
    'df_preprocesado_airbnb_temp': PROCESSED_DIR / 'df_preprocesado_airbnb_temp_v1.csv',
}

missing = [str(p.relative_to(REPO_ROOT)) for p in PROCESSED_SOURCES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos procesados. Ejecutar primero 02_normalizacion_y_limpieza.ipynb: ' + ', '.join(missing))

preprocesados = {}
for name, path in PROCESSED_SOURCES.items():
    preprocesados[name] = pd.read_csv(path, low_memory=False)
    globals()[name] = preprocesados[name]
    print(name, preprocesados[name].shape)

# Estrategia a utilizar
Se crean flags para cada variable con nulos, identificando si la columna es nula en cada fila

Luego se comparan estos flags contra variables cuantitaitivas y cualitativas para analizar posibles correlaciones. 

## Funciones para flags y gráficos de faltantes

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False


def add_null_flags(df, exclude=None):
    out = df.copy()
    exclude = set(exclude or [])
    original_cols = [c for c in out.columns if c not in exclude and not c.endswith('_is_null')]
    for col in original_cols:
        if out[col].isna().any():
            out[f'{col}_is_null'] = out[col].isna().astype('int8')
    return out


def seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=18):
    flag_cols = [c for c in df.columns if c.endswith('_is_null')]
    if not flag_cols:
        return []
    pct = df[flag_cols].mean().mul(100).sort_values(ascending=False)
    return pct[(pct >= min_pct) & (pct <= max_pct)].head(max_flags).index.tolist()


def seleccionar_cuantitativas_informativas(df, max_vars=14):
    candidatas_prioritarias = [
        'precio', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado',
        'superficie_ref_m2', 'superficie_min_m2', 'superficie_max_m2', 'superficie_total_m2',
        'superficie_cubierta_m2', 'ambientes_ref', 'ambientes', 'ambientes_min', 'ambientes_max',
        'dormitorios_ref', 'dormitorios', 'banios_ref', 'banios', 'comuna', 'altura',
        'latitud', 'longitud', 'rating', 'reviews', 'cantidad_imagenes', 'capacidad', 'camas'
    ]
    cols = []
    for col in candidatas_prioritarias:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        if df[col].notna().mean() < 0.05:
            continue
        if df[col].nunique(dropna=True) <= 1:
            continue
        cols.append(col)
    return cols[:max_vars]


def plot_heatmap_matriz(data, title, figsize=(12, 8), annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, mostrar_nan_como_cero=True):
    if data.empty:
        print(f'No hay datos suficientes para graficar: {title}')
        return
    data_plot = data.copy()
    if mostrar_nan_como_cero:
        data_plot = data_plot.fillna(0)
    plt.figure(figsize=figsize)
    if HAS_SEABORN:
        sns.heatmap(data_plot, annot=annot, fmt=fmt, cmap=cmap, center=center, vmin=vmin, vmax=vmax, linewidths=0.5, linecolor='white', cbar_kws={'label': 'Correlacion'})
    else:
        plt.imshow(data_plot, cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar(label='Correlacion')
        plt.xticks(range(len(data_plot.columns)), data_plot.columns, rotation=90)
        plt.yticks(range(len(data_plot.index)), data_plot.index)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10):
    flag_cols = [c for c in df.columns if c.endswith('_is_null')]
    if not flag_cols:
        return []
    resumen = df[flag_cols].mean().mul(100).sort_values(ascending=False).reset_index()
    resumen.columns = ['flag', 'pct_faltante']
    resumen = resumen[(resumen['pct_faltante'] >= min_pct) & (resumen['pct_faltante'] <= max_pct)]
    return resumen.head(top_n)['flag'].tolist()


def seleccionar_categoricas_informativas(df, max_cols=6):
    candidatas_prioritarias = [
        'fuente_norm', 'tipo_operacion_norm', 'moneda_norm', 'barrio_norm', 'tipo_propiedad',
        'localidad', 'provincia', 'inmobiliaria', 'seller_superhost', 'precio_sospechoso',
        'unidad_precio', 'precio_regimen', 'categoria_alojamiento', 'room_type'
    ]
    cols = []
    for col in candidatas_prioritarias:
        if col not in df.columns:
            continue
        n_labels = df[col].nunique(dropna=True)
        pct_no_nulo = df[col].notna().mean()
        if pct_no_nulo < 0.05:
            continue
        if n_labels < 2 or n_labels > 80:
            continue
        cols.append(col)
    return cols[:max_cols]


def faltantes_por_labels_categoricos(df, cat_cols, flag_cols, dataset_name, min_count=30, top_labels_por_col=25):
    rows = []
    for cat_col in cat_cols:
        if cat_col not in df.columns:
            continue
        serie_cat = df[cat_col].astype('string').fillna(f'NULO_EN_{cat_col}')
        labels_validos = serie_cat.value_counts(dropna=False).head(top_labels_por_col).index
        for label in labels_validos:
            mask = serie_cat == label
            n = int(mask.sum())
            if n < min_count:
                continue
            for flag_col in flag_cols:
                if flag_col not in df.columns:
                    continue
                if flag_col == f'{cat_col}_is_null':
                    continue
                rows.append({
                    'dataset': dataset_name,
                    'variable_categorica': cat_col,
                    'label': label,
                    'n_registros': n,
                    'variable_faltante': flag_col,
                    'pct_faltante': round(df.loc[mask, flag_col].mean() * 100, 2),
                })
    return pd.DataFrame(rows)


def graficar_faltantes_por_label(df_resumen, dataset_name):
    if df_resumen.empty:
        print(f'No se genero tabla de faltantes por label para {dataset_name}.')
        return
    df_pivot = df_resumen.pivot_table(
        index=['variable_categorica', 'label'],
        columns='variable_faltante',
        values='pct_faltante',
        aggfunc='mean'
    ).fillna(0)
    if df_pivot.empty:
        print(f'No hay datos suficientes para graficar {dataset_name}.')
        return
    variables_unicas = list(df_pivot.index.get_level_values('variable_categorica').unique())
    alto = min(22, max(7, len(df_pivot) * 0.32))
    if HAS_SEABORN:
        paleta = sns.color_palette('Set2', n_colors=len(variables_unicas))
        mapa_colores = dict(zip(variables_unicas, paleta))
        colores_filas = [mapa_colores[v] for v in df_pivot.index.get_level_values('variable_categorica')]
        g = sns.clustermap(
            df_pivot,
            row_cluster=False,
            col_cluster=False,
            row_colors=colores_filas,
            cmap='YlOrRd',
            annot=True,
            fmt='.1f',
            linewidths=0.5,
            figsize=(14, alto),
        )
        g.fig.suptitle(f'Porcentaje de faltantes por label categorico - {dataset_name}', y=1.02)
        plt.show()
    else:
        plt.figure(figsize=(14, alto))
        plt.imshow(df_pivot.values, aspect='auto')
        plt.colorbar(label='% faltante')
        plt.xticks(range(len(df_pivot.columns)), df_pivot.columns, rotation=90)
        plt.yticks(range(len(df_pivot.index)), [f'{a} = {b}' for a, b in df_pivot.index], fontsize=8)
        plt.title(f'Porcentaje de faltantes por label categorico - {dataset_name}')
        plt.tight_layout()
        plt.show()


## Creación de flags de nulidad por dataset original

In [ ]:
preprocesados_con_flags = {}
for name, df in preprocesados.items():
    preprocesados_con_flags[name] = add_null_flags(df)
    globals()[name] = preprocesados_con_flags[name]
    print(name, 'flags creados:', sum(c.endswith('_is_null') for c in preprocesados_con_flags[name].columns))

## Celdas de decisión: estrategia inicial de nulos

Antes de aplicar ajustes, se conserva una estrategia conservadora: flaggear y analizar. La imputación o eliminación se decide después de revisar los patrones.

In [ ]:
# Estrategia conservadora por defecto.
# Opciones esperadas: 'flaggear', 'eliminar_criticos'
ESTRATEGIA_NULOS = 'flaggear'
CAMPOS_CRITICOS_NULOS = ['precio', 'moneda_norm', 'barrio_norm', 'tipo_operacion_norm']

resumen_estrategia_nulos = []
for name, df in preprocesados_con_flags.items():
    campos_existentes = [c for c in CAMPOS_CRITICOS_NULOS if c in df.columns]
    filas_antes = len(df)
    filas_con_nulo_critico = int(df[campos_existentes].isna().any(axis=1).sum()) if campos_existentes else 0

    if ESTRATEGIA_NULOS == 'eliminar_criticos' and campos_existentes:
        preprocesados_con_flags[name] = df.dropna(subset=campos_existentes).copy()
        accion = 'eliminados registros con nulos en campos criticos'
    else:
        accion = 'solo flags; no se eliminan registros'

    resumen_estrategia_nulos.append({
        'dataset': name,
        'estrategia': ESTRATEGIA_NULOS,
        'campos_criticos_existentes': ', '.join(campos_existentes),
        'filas_antes': filas_antes,
        'filas_con_nulo_critico': filas_con_nulo_critico,
        'filas_despues': len(preprocesados_con_flags[name]),
        'accion': accion,
    })

resumen_estrategia_nulos = pd.DataFrame(resumen_estrategia_nulos)
display(resumen_estrategia_nulos)
resumen_estrategia_nulos.to_csv(REPORTS_DIR / 'decisiones_tratamiento_nulos_v1.csv', index=False)

## Matrices de faltantes por dataset

Tal como hicimos en el caso practico de las clases del 23 y 24 de septiembre, se muestran dos matrices de correlaciones por dataset:

1. **Faltante vs faltante**, para ver ausencias que aparecen juntas.
2. **Faltante vs cuantitativas**, para ver si la ausencia de un campo se asocia con precio, superficie, ambientes, rating u otras variables numéricas.

Estas matrices permiten analizar si los faltantes se correlacionan con otros faltantes o con variables cuantitativas. 



In [ ]:
for name, df in preprocesados_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags = seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=16)
    cuantitativas = seleccionar_cuantitativas_informativas(df, max_vars=12)

    if len(flags) < 2:
        print('No hay suficientes flags informativos de faltantes para este dataset.')
        continue

    matriz_faltante_faltante = df[flags].corr()
    matriz_faltante_faltante.to_csv(REPORTS_DIR / f'matriz_faltante_vs_faltante_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_faltante, title=f'Faltante vs faltante - {name}', figsize=(12, 9), annot=True, mostrar_nan_como_cero=True)

    if len(cuantitativas) == 0:
        print('No hay variables cuantitativas informativas para cruzar contra faltantes.')
        continue

    matriz_faltante_cuantitativa = df[flags + cuantitativas].corr(numeric_only=True).loc[flags, cuantitativas]
    umbral_corr_cuantitativas = 0.15
    matriz_faltante_cuantitativa_filtrada = matriz_faltante_cuantitativa.where(matriz_faltante_cuantitativa.abs() >= umbral_corr_cuantitativas, other=0)
    matriz_faltante_cuantitativa.to_csv(REPORTS_DIR / f'matriz_faltante_vs_cuantitativas_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_cuantitativa_filtrada, title=f'Faltante vs variables cuantitativas - {name}', figsize=(13, 8), annot=True, mostrar_nan_como_cero=True)

# Conclusiones de correlaciones originales

### MeLi alquiler
Se nota un bloque de correlaciones fuertes en variables de rangos de ambientes, mínimos y máximos. Esto sugiere que varias ausencias no son aleatorias sino que dependen de cómo la publicación estructura los atributos.

### Argenprop y Zonaprop combinados
En esta primera lectura aparecen mezcladas publicaciones de venta y alquiler dentro de una misma fuente. Eso puede inflar o esconder relaciones entre faltantes, porque una operación puede tener campos naturalmente más presentes que otra. Por eso se decide generar una segunda lectura separando por `tipo_operacion_norm`.

### Las Cañitas
En las matrices y tablas por label aparece `Las Cañitas / las canitas` como barrio operativo con comuna faltante. Como no es barrio oficial de CABA sino subzona de Palermo, se decide corregirlo **después** de observar el problema: normalizarlo como `palermo` y completar `comuna = 14`.


## Faltantes por label de variables categóricas, discriminado por dataset

Este análisis evita que el consolidado quede dominado por diferencias de estructura entre fuentes. Para cada dataset se cruzan flags de faltantes contra labels de variables categóricas existentes en ese mismo dataset.

In [ ]:
resumenes_faltantes_por_label = []
for name, df in preprocesados_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags_diagnostico = seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10)
    cat_cols_diagnostico = seleccionar_categoricas_informativas(df, max_cols=6)

    if not flags_diagnostico:
        print('No hay flags de faltantes informativos para este dataset.')
        continue
    if not cat_cols_diagnostico:
        print('No hay variables categoricas informativas para este dataset.')
        continue

    df_faltantes_label_dataset = faltantes_por_labels_categoricos(df, cat_cols_diagnostico, flags_diagnostico, dataset_name=name, min_count=30, top_labels_por_col=25)
    if df_faltantes_label_dataset.empty:
        print('No hay combinaciones con volumen suficiente para graficar.')
        continue

    resumenes_faltantes_por_label.append(df_faltantes_label_dataset)
    df_faltantes_label_dataset.to_csv(REPORTS_DIR / f'faltantes_por_label_categorico_{name}.csv', index=False)
    graficar_faltantes_por_label(df_faltantes_label_dataset, dataset_name=name)

if resumenes_faltantes_por_label:
    df_faltantes_por_label_todos = pd.concat(resumenes_faltantes_por_label, ignore_index=True)
    df_faltantes_por_label_todos.to_csv(REPORTS_DIR / 'faltantes_por_label_categorico_todos_los_datasets.csv', index=False)
    display(df_faltantes_por_label_todos.head(50))

# Conclusiones de faltantes por labels originales

Los faltantes se distribuyen de manera relativamente uniforme en varios barrios, pero aparecen excepciones útiles para decidir tratamiento. En particular, `Las Cañitas` muestra un faltante artificial de comuna por criterio de normalización geográfica. Además, Argenprop y Zonaprop todavía están combinados por fuente, por lo que la lectura por label se toma como diagnóstico inicial y no como cierre.

A partir de este punto se hacen dos ajustes controlados y luego se vuelven a mirar matrices/tablas:

1. Separar Argenprop y Zonaprop por tipo de operación.
2. Normalizar `Las Cañitas / las canitas` como Palermo y completar comuna 14.


In [ ]:
# Guardado del diagnostico original con flags.
# Estos archivos conservan el estado previo a los ajustes decididos a partir de las matrices.
for name, df in preprocesados_con_flags.items():
    path = PROCESSED_DIR / f'{name}_con_flags_nulos_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado diagnostico original:', path.relative_to(REPO_ROOT), df.shape)


## Ajustes derivados del diagnóstico original

Estos ajustes no reemplazan el diagnóstico anterior: aparecen después porque son consecuencia de lo observado en matrices y tablas por label.

- `Las Cañitas / las canitas` se corrige como subzona de Palermo y se completa `comuna = 14`.
- Argenprop y Zonaprop se separan en venta y alquiler para que los faltantes estructurales de cada operación no queden mezclados.


In [ ]:
BARRIO_COMUNA = {
    'agronomia': 15, 'almagro': 5, 'balvanera': 3, 'barracas': 4,
    'belgrano': 13, 'boedo': 5, 'caballito': 6, 'chacarita': 15,
    'coghlan': 12, 'colegiales': 13, 'constitucion': 1, 'flores': 7,
    'floresta': 10, 'la boca': 4, 'liniers': 9, 'mataderos': 9,
    'monserrat': 1, 'monte castro': 10, 'nueva pompeya': 4,
    'nunez': 13, 'palermo': 14, 'parque avellaneda': 9,
    'parque chacabuco': 7, 'parque chas': 15,
    'parque patricios': 4, 'paternal': 15, 'puerto madero': 1,
    'recoleta': 2, 'retiro': 1, 'saavedra': 12,
    'san cristobal': 3, 'san nicolas': 1, 'san telmo': 1,
    'velez sarsfield': 10, 'versalles': 10,
    'villa crespo': 15, 'villa del parque': 11, 'villa devoto': 11,
    'villa general mitre': 11, 'villa lugano': 8, 'villa luro': 10,
    'villa ortuzar': 15, 'villa pueyrredon': 12, 'villa real': 10,
    'villa riachuelo': 8, 'villa santa rita': 11,
    'villa soldati': 8, 'villa urquiza': 12,
}

BARRIO_ALIASES_POST_DIAGNOSTICO = {
    'las cañitas': 'palermo',
    'las canitas': 'palermo',
    'palermo hollywood': 'palermo',
    'palermo soho': 'palermo',
    'palermo chico': 'palermo',
}


def aplicar_ajustes_post_diagnostico(df):
    out = df.copy()
    if 'barrio_norm' in out.columns:
        barrio_norm = out['barrio_norm'].astype('string').str.strip().str.lower()
        out['barrio_norm'] = barrio_norm.replace(BARRIO_ALIASES_POST_DIAGNOSTICO)
    if 'barrio' in out.columns and 'barrio_norm' in out.columns:
        barrio_original = out['barrio'].astype('string').str.strip().str.lower()
        mask_canitas = barrio_original.isin(['las cañitas', 'las canitas'])
        out.loc[mask_canitas, 'barrio_norm'] = 'palermo'
    if 'comuna' in out.columns and 'barrio_norm' in out.columns:
        out['comuna'] = pd.to_numeric(out['comuna'], errors='coerce')
        comuna_desde_barrio = out['barrio_norm'].map(BARRIO_COMUNA)
        mask_comuna = out['comuna'].isna() & comuna_desde_barrio.notna()
        out.loc[mask_comuna, 'comuna'] = comuna_desde_barrio.loc[mask_comuna]
    return out


def resumen_canitas(df, dataset, momento):
    if 'barrio_norm' not in df.columns:
        return None
    mask = df['barrio_norm'].astype('string').str.contains('canit|cañit', case=False, na=False)
    comuna_nula = int(df.loc[mask, 'comuna'].isna().sum()) if 'comuna' in df.columns else np.nan
    return {
        'dataset': dataset,
        'momento': momento,
        'filas_las_canitas_en_barrio_norm': int(mask.sum()),
        'comuna_nula_en_las_canitas': comuna_nula,
    }

resumen_ajuste_canitas = []
preprocesados_post_diagnostico = {}
for name, df in preprocesados.items():
    r_antes = resumen_canitas(df, name, 'antes')
    if r_antes:
        resumen_ajuste_canitas.append(r_antes)
    ajustado = aplicar_ajustes_post_diagnostico(df)
    preprocesados_post_diagnostico[name] = ajustado
    r_despues = resumen_canitas(ajustado, name, 'despues')
    if r_despues:
        resumen_ajuste_canitas.append(r_despues)

resumen_ajuste_canitas = pd.DataFrame(resumen_ajuste_canitas)
display(resumen_ajuste_canitas)
resumen_ajuste_canitas.to_csv(REPORTS_DIR / 'ajuste_canitas_post_diagnostico_v1.csv', index=False)


def separar_por_operacion(nombre_base, df, mapeo_operaciones):
    resultado = {}
    categorias_presentes = set(df['tipo_operacion_norm'].dropna().unique())
    categorias_sin_mapear = categorias_presentes - set(mapeo_operaciones)
    if categorias_sin_mapear:
        print(f'AVISO [{nombre_base}]: categorias sin mapear, se excluyen: {categorias_sin_mapear}')
    for valor_operacion, sufijo in mapeo_operaciones.items():
        sub_df = df[df['tipo_operacion_norm'] == valor_operacion].copy()
        clave = f'{nombre_base}_{sufijo}'
        resultado[clave] = sub_df
        print(clave, sub_df.shape)
    return resultado

MAPEO_OPERACIONES_PORTALES = {'venta': 'vtas', 'alquiler': 'alq'}

preprocesados_post_ajustes = {
    name: df for name, df in preprocesados_post_diagnostico.items()
    if name not in {'df_preprocesado_argenprop', 'df_preprocesado_zonaprop'}
}
preprocesados_post_ajustes.update(separar_por_operacion('df_preprocesado_argenprop', preprocesados_post_diagnostico['df_preprocesado_argenprop'], MAPEO_OPERACIONES_PORTALES))
preprocesados_post_ajustes.update(separar_por_operacion('df_preprocesado_zonaprop', preprocesados_post_diagnostico['df_preprocesado_zonaprop'], MAPEO_OPERACIONES_PORTALES))

preprocesados_post_ajustes_con_flags = {}
for name, df in preprocesados_post_ajustes.items():
    preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
    globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]

print('\nDatasets post-ajustes:')
for name, df in preprocesados_post_ajustes_con_flags.items():
    print(name, df.shape)


## Matrices post-ajustes para comparar

Estas matrices son una segunda lectura. No eliminan las matrices originales: sirven para verificar si las decisiones tomadas reducen faltantes artificiales o separan patrones que estaban mezclados.

In [ ]:
for name, df in preprocesados_post_ajustes_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags = seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=16)
    cuantitativas = seleccionar_cuantitativas_informativas(df, max_vars=12)

    if len(flags) < 2:
        print('No hay suficientes flags informativos de faltantes para este dataset.')
        continue

    matriz_faltante_faltante = df[flags].corr()
    matriz_faltante_faltante.to_csv(REPORTS_DIR / f'matriz_faltante_vs_faltante_post_ajustes_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_faltante, title=f'POST AJUSTES - Faltante vs faltante - {name}', figsize=(12, 9), annot=True, mostrar_nan_como_cero=True)

    if len(cuantitativas) == 0:
        print('No hay variables cuantitativas informativas para cruzar contra faltantes.')
        continue

    matriz_faltante_cuantitativa = df[flags + cuantitativas].corr(numeric_only=True).loc[flags, cuantitativas]
    umbral_corr_cuantitativas = 0.15
    matriz_faltante_cuantitativa_filtrada = matriz_faltante_cuantitativa.where(matriz_faltante_cuantitativa.abs() >= umbral_corr_cuantitativas, other=0)
    matriz_faltante_cuantitativa.to_csv(REPORTS_DIR / f'matriz_faltante_vs_cuantitativas_post_ajustes_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_cuantitativa_filtrada, title=f'POST AJUSTES - Faltante vs variables cuantitativas - {name}', figsize=(13, 8), annot=True, mostrar_nan_como_cero=True)


## Faltantes por label post-ajustes

Se repite el cruce contra variables categóricas para verificar si los labels problemáticos siguen mostrando faltantes concentrados después de separar operaciones y corregir el caso geográfico detectado.

In [ ]:
resumenes_faltantes_por_label_post = []
for name, df in preprocesados_post_ajustes_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags_diagnostico = seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10)
    cat_cols_diagnostico = seleccionar_categoricas_informativas(df, max_cols=6)

    if not flags_diagnostico:
        print('No hay flags de faltantes informativos para este dataset.')
        continue
    if not cat_cols_diagnostico:
        print('No hay variables categoricas informativas para este dataset.')
        continue

    df_faltantes_label_dataset = faltantes_por_labels_categoricos(df, cat_cols_diagnostico, flags_diagnostico, dataset_name=name, min_count=30, top_labels_por_col=25)
    if df_faltantes_label_dataset.empty:
        print('No hay combinaciones con volumen suficiente para graficar.')
        continue

    resumenes_faltantes_por_label_post.append(df_faltantes_label_dataset)
    df_faltantes_label_dataset.to_csv(REPORTS_DIR / f'faltantes_por_label_categorico_post_ajustes_{name}.csv', index=False)
    graficar_faltantes_por_label(df_faltantes_label_dataset, dataset_name=f'POST AJUSTES - {name}')

if resumenes_faltantes_por_label_post:
    df_faltantes_por_label_post_todos = pd.concat(resumenes_faltantes_por_label_post, ignore_index=True)
    df_faltantes_por_label_post_todos.to_csv(REPORTS_DIR / 'faltantes_por_label_categorico_post_ajustes_todos_los_datasets.csv', index=False)
    display(df_faltantes_por_label_post_todos.head(50))


## Guardado de versiones post-ajustes

In [ ]:
# Guardado post-ajustes. Se usa otro sufijo para no pisar el diagnostico original.
for name, df in preprocesados_post_ajustes_con_flags.items():
    path = PROCESSED_DIR / f'{name}_post_ajustes_con_flags_nulos_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado post-ajustes:', path.relative_to(REPO_ROOT), df.shape)


## Cierre del análisis de nulos

El notebook conserva dos momentos:

1. **Diagnóstico original:** matrices y tablas sobre los datasets tal como llegan desde la normalización inicial.
2. **Post-ajustes:** matrices y tablas después de decisiones tomadas a partir del diagnóstico: separación de Argenprop/Zonaprop por operación y corrección de Las Cañitas como Palermo.

Esta estructura deja trazabilidad metodológica: no se oculta el problema, se muestra cómo se detectó y por qué se decidió corregirlo.

## Celdas finales para decidir tratamiento por columna

Estas celdas no aplican todavía imputaciones masivas. Funcionan como bitácora de decisión: `columna`, `hacer`, `motivo` y `estado`. La idea es completar o ajustar esta tabla antes de pasar al tratamiento definitivo.

In [ ]:
decisiones_manual_nulos = pd.DataFrame([
    {'columna': 'precio', 'hacer': 'no imputar; excluir solo de KPIs de precio si falta', 'motivo': 'variable objetivo/central; inventarla distorsiona todo indicador monetario', 'estado': 'propuesto'},
    {'columna': 'moneda_norm', 'hacer': 'no imputar; reconstruir solo si precio_texto trae simbolo claro', 'motivo': 'sin moneda no se puede comparar precio ni convertir ARS/USD', 'estado': 'propuesto'},
    {'columna': 'barrio_norm', 'hacer': 'reconstruir desde barrio/direccion/coordenadas; si no, excluir de analisis geografico', 'motivo': 'es clave para hipotesis territoriales y cruces con fuentes externas', 'estado': 'propuesto'},
    {'columna': 'comuna', 'hacer': 'recalcular desde barrio_norm cuando exista mapeo oficial', 'motivo': 'es una variable derivable; no conviene dejar faltantes artificiales como Las Cañitas', 'estado': 'propuesto'},
    {'columna': 'superficie_ref_m2', 'hacer': 'reconstruir desde superficies equivalentes o texto explicito; no imputar media', 'motivo': 'define precio_m2 y tipologia; una imputacion promedio altera el inmueble', 'estado': 'propuesto'},
    {'columna': 'precio_m2_ref', 'hacer': 'recalcular cuando haya precio y superficie_ref_m2 validos', 'motivo': 'es derivada, no observada; debe seguir la calidad de sus componentes', 'estado': 'propuesto'},
    {'columna': 'ambientes_ref', 'hacer': 'reconstruir desde min/max o texto explicito', 'motivo': 'la cantidad de ambientes define comparabilidad; no inferir sin evidencia', 'estado': 'propuesto'},
    {'columna': 'dormitorios_ref', 'hacer': 'mantener nulo salvo mencion explicita; evaluar regla especial para monoambientes', 'motivo': 'muchos portales publican ambientes y no dormitorios; no son equivalentes directos', 'estado': 'propuesto'},
    {'columna': 'banios_ref', 'hacer': 'reconstruir desde min/max, atributos o texto explicito', 'motivo': 'variable estructural relevante; no imputar moda', 'estado': 'propuesto'},
    {'columna': 'latitud/longitud', 'hacer': 'no imputar coordenada real; si se usan centroides, crear columnas *_aprox', 'motivo': 'mezclar coordenadas reales y aproximadas rompe la interpretabilidad espacial', 'estado': 'propuesto'},
    {'columna': 'inmobiliaria/seller', 'hacer': 'usar no_informado solo para graficos; mantener flag para analisis', 'motivo': 'el faltante puede ser informativo del portal o tipo de publicacion', 'estado': 'propuesto'},
    {'columna': 'expensas', 'hacer': 'parsear desde texto si aparece; no reemplazar por 0', 'motivo': 'no informado no significa sin expensas', 'estado': 'propuesto'},
    {'columna': 'rating/reviews', 'hacer': 'mantener nulos y flag sin_reviews_o_rating; no imputar promedio', 'motivo': 'en Airbnb el faltante puede indicar ausencia de reseñas y no calidad media', 'estado': 'propuesto'},
    {'columna': 'estado_entrega/posesion', 'hacer': 'usar como variable secundaria; reconstruir desde texto si aparece', 'motivo': 'muy incompleta, pero relevante en ventas/emprendimientos', 'estado': 'propuesto'},
    {'columna': 'Airbnb superficie/precio_m2', 'hacer': 'no forzar precio_m2; analizar Airbnb con precio mensual/noche', 'motivo': 'Airbnb no trae superficie comparable; imputarla seria artificial', 'estado': 'propuesto'},
])

display(decisiones_manual_nulos)
decisiones_manual_nulos.to_csv(REPORTS_DIR / 'decisiones_manual_tratamiento_nulos_v1.csv', index=False)


### Celda de ejecución controlada

In [ ]:
APLICAR_TRATAMIENTO_NULOS = False

if APLICAR_TRATAMIENTO_NULOS:
    # Completar aca solo reglas ya aprobadas en decisiones_manual_nulos.
    # Ejemplos:
    # - df['precio_m2_ref'] = df['precio'] / df['superficie_ref_m2'] cuando ambas existan y sean validas
    # - crear texto_analisis = titulo + descripcion + texto_tarjeta
    # - crear latitud_aprox/longitud_aprox desde centroides, sin pisar coordenadas reales
    raise NotImplementedError('Completar reglas finales antes de activar el tratamiento.')
else:
    print('Tratamiento no aplicado: queda documentada la decision por columna y se conservan flags para auditoria.')
